# ESI heater characterization — up to 100 °C
**Active, supervised test of the original CGC capillary/heater. No HV test.**

Close Explorer and CGC utilities. Keep `notebooks/` beside the `esi/` plugin folder (or set `PLUGIN_DIR`), use a fresh **64-bit Windows** kernel, check `ESI_COM`, and run the one cell. **`ARM_HEATING=True` by default: running the cell requests the real heater experiment.** Set it to `False` for a no-hardware check. Existing unfinished-run guards still block activation. Keep the physical shutdown controls accessible; check any cooling required by your assembly.

- Fixed envelope **≤22 V / 10 A / 50 W**, also bounded by fresh hardware limits; applied values are quantized downward and independently read back. The 22 V / 50 W setting is present in CGC's Heat60 profile; no complete profile is loaded; 10 A retains the controller's nominal designation instead of the profiles' 12 A. **These are test settings, not certified load ratings. No automatic power increase.**
- Start below 30 °C. Baseline OFF **60 s**; **30 → 40 → 50 → 60 → 70 → 80 → 90 °C**. First reach ±1 °C within **300 s**, then complete **60 continuous seconds** in that zone, with a fixed **360 s total limit**. An excursion resets observation, never extends that deadline. At the first successful 30 °C stage: confirmed heater OFF for 60 s, verify unchanged electrical limits, then repeat 30 °C. No reconnection or hidden limit rewrite.
- Final target **100 °C**, limited to **300 s with no extension**: OFF at the first observation ≥100 °C, **no hold at 100 °C**. A sampled software stop cannot prevent all overshoot/inertia. Non-attainment stops the run; it does not prove a fault or define an admissible power.
- After confirmed heater OFF, observe cooling for **180 s**, then use the runtime's verified HV discharge and port closure. This does not establish that the heater is cold or safe to touch.

**Stop:** interrupt the kernel once. The single instrument owner waits for the current call before OFF; never launch another cell/Close concurrently. A second Stop can shorten cooling observation, not discharge verification. A poisoned/unknown transport blocks all further ESI commands: use physical safety controls. An unfinished-run marker and memory guard block rerun; restarting a kernel is not an OFF.

This revision records existing native-wrapper calls and discharge observations in `native_calls.jsonl`, including statuses, Valid flags, non-finite values, and start/return timestamps. Tracing adds no instrument calls or retry. The matched runtime now waits for the heater MON_RDY bit before one monitoring read, within the existing I/O budget; Stop cancels this wait and pending heater activation, not cleanup. Invalid data after readiness still refuse heating. Logging has timing overhead and is not a raw serial capture. The previous unfinished-run marker remains blocking; this revision provides no operator-recovery bypass.

Electrical limits **remain configured after the test**; initial zeros are not restored. No Save/NVM command is issued; persistence after power cycling is unknown. **This test does not qualify 175 °C**, even if the positive limits subsequently satisfy another notebook's preflight.

CSV/JSON and `heater.png` are saved under `esi/logs/esi_heater_characterization/` in the selected plugin folder. Existing logs and unfinished-run markers stay there; do not move or delete them. VoltOut, VoltHeat and CurrOut are separate monitoring channels; voltage/power targets are not measurements. Calculated VoltHeat × CurrOut is **not validated dissipated power** (PWM/averaging/synchronization unknown); no undocumented U/I-to-limit alarm is claimed. Recording is best effort ~1 Hz with actual call times, not certified fresh/synchronous ADC samples. Reliable electrical power/energy characterization needs CGC clarification or suitable external instrumentation.

**Operator restart after an unfinished run:** in a fresh kernel, the notebook requires one explicit, run-specific typed declaration that the equipment is physically safe and the previous ESI/pressure-owning processes have terminated. This is an operator declaration, not software verification or confirmation of HV discharge. The characterization and pressure–temperature notebooks share a per-user COM registry/lock under `~/.esibd/esi_experiment_guards/` and the historical heater lock. Original evidence is archived there with hashes before updating the central claim and compatibility markers. Interrupted file transitions remain blocking; they are not a single atomic transaction. No claim is released before the final report is durable and shutdown/owners are confirmed. These guards do not cover other users, physical COM aliases or unknown legacy output folders. Missing original evidence (e.g. a deleted run folder) is listed and archived before the same declaration; changed evidence or an ownership conflict blocks the experiment. ARM alone never acknowledges recovery.


In [ ]:
# Run from notebooks/ beside esi/ with Explorer and CGC utilities CLOSED.
ARM_HEATING = True  # Running this cell requests the approved real heater test.
PLUGIN_DIR = None  # Optional esi folder; normally found beside the notebooks folder.
ESI_COM = 16
ESI_BAUD = 230400

import csv
import hashlib
import functools
import importlib.util
import importlib.machinery
import json
import math
import os
from pathlib import Path
import queue
import signal
import sys
import threading
import time
from datetime import datetime, timezone
from uuid import uuid4

NOTEBOOK_DIR = Path(globals().get('__file__', Path.cwd() / 'esi_heater_characterization.ipynb')).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / 'logs' / 'esi_heater_characterization'
_HC_GUARD = globals().get('_HC_GUARD', {})


def explorer_plugin_dirs():
    '''ESIBD Explorer's configured plugin folder (Windows settings), then its default.'''
    found = []
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Software\ESIBD LAB\ESIBD Explorer\General') as key:
            value = winreg.QueryValueEx(key, 'Plugin path')[0]
        if isinstance(value, str) and value and not value.startswith('@'):
            found.append(Path(value).expanduser())
    except (ImportError, OSError):
        pass
    found.append(Path.home() / 'ESIBD Explorer' / 'plugins')
    return found


def find_plugin(configured=None):
    candidates = [Path(configured).expanduser().resolve()] if configured is not None else [
        item for parent in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) for item in (parent, parent / 'esi', parent / 'plugins' / 'esi')
    ] + [plugins / 'esi' for plugins in explorer_plugin_dirs()]
    for candidate in candidates:
        if (candidate / 'vendor/runtime/__init__.py').is_file():
            return candidate
    raise FileNotFoundError("esi/ not found beside the notebook nor in ESIBD Explorer's plugin folder; "
                            'set PLUGIN_DIR to the folder containing esi_plugin.py.')


# Recorded with each run (provenance only; any plugin version may be used).
_RUNTIME_FILES = ('vendor/runtime/__init__.py', 'vendor/runtime/_controller_process.py', 'vendor/runtime/_driver_common.py', 'vendor/runtime/_process_proxy_support.py', 'vendor/runtime/esi/__init__.py', 'vendor/runtime/esi/esi.py', 'vendor/runtime/esi/esi_base.py', 'vendor/runtime/esi/vendor/x64/COM-ESI-CTRL.dll', '_experiment_guard.py', '_heater_limits.py')

def utc(timestamp=None):
    return datetime.fromtimestamp(time.time() if timestamp is None else timestamp, timezone.utc).isoformat()


def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value


def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(json_safe(data), stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def load_private(path, package=False, fresh=False):
    path = Path(path).resolve()
    runtime_root = path.parent
    for loaded in tuple(sys.modules.values()):
        source = getattr(loaded, '__file__', None)
        if source:
            try:
                if Path(source).resolve().is_relative_to(runtime_root):
                    raise RuntimeError('ESI runtime already present (possibly stale). Confirm hardware OFF, then use a fresh kernel; never bypass an unfinished-run guard.')
            except (OSError, ValueError, TypeError):
                pass
    digest = hashlib.sha256(os.path.normcase(str(path)).encode()).hexdigest()[:16]
    name = "_esibd_bundled_heater_characterization_" + digest
    if fresh and any(key == name or key.startswith(name + ".") for key in sys.modules):
        raise RuntimeError("ESI runtime already loaded. After confirmed hardware OFF, restart the kernel; never bypass an unfinished-run guard.")
    sources = sorted(path.parent.rglob("*.py")) if package else [path]
    fingerprint = {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
    if name in sys.modules and getattr(sys.modules[name], "_source_fingerprint", None) != fingerprint:
        raise RuntimeError("Bundled runtime changed after import. Confirm hardware OFF, then restart the kernel.")
    if name not in sys.modules:
        class SourceLoader(importlib.machinery.SourceFileLoader):
            def get_code(self, fullname):
                source = self.get_data(self.path)
                if hashlib.sha256(source).hexdigest() != fingerprint.get(os.path.normcase(str(Path(self.path).resolve()))):
                    raise RuntimeError("Runtime changed during import; restart the kernel.")
                return self.source_to_code(source, self.path)  # Compile verified bytes, never a stale .pyc.

        def source_spec(fullname, file, is_package):
            return importlib.util.spec_from_file_location(
                fullname, file, loader=SourceLoader(fullname, str(file)),
                submodule_search_locations=[str(file.parent)] if is_package else None)

        class PrivateFinder:
            def find_spec(self, fullname, path=None, target=None):
                if not fullname.startswith(name + "."):
                    return None
                relative = fullname[len(name) + 1:].split(".")
                file = package_root.joinpath(*relative)
                is_package = file.is_dir()
                file = file / "__init__.py" if is_package else file.with_suffix(".py")
                return source_spec(fullname, file, is_package) if file.is_file() else None

        package_root = path.parent
        finder = PrivateFinder()
        spec = source_spec(name, path, package)
        if spec is None or spec.loader is None:
            raise ModuleNotFoundError(str(path))
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            sys.meta_path.insert(0, finder)  # Restricted to this new private namespace; sys.path is untouched.
            spec.loader.exec_module(module)
            if fingerprint != {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}:
                raise RuntimeError("Runtime changed during import; restart the kernel.")
            module._source_fingerprint = fingerprint
        except BaseException:
            # Keep a failed-load marker: do not reuse partially imported package children.
            module._source_fingerprint = None
            raise
        finally:
            if finder in sys.meta_path:
                sys.meta_path.remove(finder)
    return sys.modules[name]


def runtime_fingerprints(root):
    """SHA-256 of the runtime files used by this run, for its report only."""
    return {path: hashlib.sha256((root / path).read_bytes()).hexdigest()
            for path in _RUNTIME_FILES if (root / path).is_file()}


_SAFE_ESI_REQUIRED = "The loaded ESI runtime lacks the heater safety methods; update the esi/ plugin."


def require_safe_esi(runtime):
    # Check the class actually imported, not merely a candidate path or method names.
    try:
        controller = runtime.ESI._PROCESS_CONTROLLER_CLASS
        valid = (controller.__module__.startswith(runtime.__name__ + ".")
                 and getattr(controller, "HEATER_SAFETY_CONTRACT", 0) >= 1
                 and all(callable(getattr(controller, name, None)) for name in (
                     "_set_heat_module_active_unlocked", "_validate_heat_operating_state_unlocked")))
    except (AttributeError, TypeError) as error:
        raise RuntimeError(_SAFE_ESI_REQUIRED) from error
    if not valid:
        raise RuntimeError(_SAFE_ESI_REQUIRED)


class NativeTrace:
    """Observe existing base-wrapper calls only; disk I/O stays off the DLL thread."""
    METHODS = ('open_port', 'close_port', 'get_heat_ctrl_monitoring',
               'get_complete_state', 'get_module_data_ready_flags',
               'get_hv_supply_meas_ranges', 'set_hv_supply_meas_ranges',
               'get_hv_supply_output_voltage', 'get_hv_supply_output_current',
               'get_heat_ctrl_heater_temperature', 'set_heat_ctrl_heater_temperature',
               'get_module_activation_state', 'set_module_activation_state', 'get_enable', 'set_enable')

    def __init__(self, directory):
        self.path = Path(directory) / 'native_calls.jsonl'
        self.stream = self.path.open('x', encoding='utf-8')
        self.queue = queue.SimpleQueue()
        self.errors, self.hooks = [], []
        self.closed = False
        self.call_ids = iter(range(sys.maxsize))
        self.thread = threading.Thread(target=self._write, name='heater-trace-writer', daemon=True)
        self.thread.start()

    @staticmethod
    def encode(value):
        # Preserve non-finite values distinctly; unlike plotting JSON, never turn them into null.
        if isinstance(value, float) and not math.isfinite(value):
            return {'nonfinite_float': repr(value)}
        if isinstance(value, dict):
            return {str(k): NativeTrace.encode(v) for k, v in value.items()}
        if isinstance(value, (tuple, list)):
            return [NativeTrace.encode(v) for v in value]
        return value

    def record(self, event, **fields):
        try:
            if self.closed:
                return
            self.queue.put(dict(event=event, monotonic_ns=time.monotonic_ns(),
                                utc_ns=time.time_ns(), thread_id=threading.get_ident(),
                                **self.encode(fields)))
        except BaseException as error:
            if len(self.errors) < 10:
                self.errors.append(type(error).__name__)  # Never substitute a logging error for a native result.

    def _write(self):
        try:
            while True:
                row = self.queue.get()
                if row is None:
                    break
                try:
                    self.stream.write(json.dumps(self.encode(row), allow_nan=False) + '\n')
                    self.stream.flush()
                except BaseException as error:
                    if len(self.errors) < 10:
                        self.errors.append(f'{type(error).__name__}: {error}')
        finally:
            try:
                self.stream.close()
            except BaseException as error:
                self.errors.append(type(error).__name__)

    def install(self, base, backend):
        originals = [(name, getattr(base, name)) for name in self.METHODS]
        if not isinstance(backend, base) or not all(callable(f) for _, f in originals):
            raise RuntimeError('Native tracing requires the matched inline ESI backend; no connection attempted.')
        def wrap(name, original):
            def observed(device, *args, **kwargs):
                if device is not backend:
                    return original(device, *args, **kwargs)
                call_id = next(self.call_ids)
                self.record('native_start', call_id=call_id, method=name, args=args, kwargs=kwargs)
                try:
                    result = original(device, *args, **kwargs)
                except BaseException as error:
                    self.record('native_exception', call_id=call_id, method=name,
                                exception_type=type(error).__name__, message=str(error))
                    raise
                self.record('native_return', call_id=call_id, method=name, result=result)
                return result  # Same object/status/Valid, including false and non-finite data.
            return functools.wraps(original)(observed)
        for name, original in originals:
            observed = wrap(name, original)
            self.hooks.append((base, name, original, observed))
            setattr(base, name, observed)

    def finish(self, native_finished):
        if native_finished and not self.closed:
            for base, name, original, observed in self.hooks:
                if getattr(base, name) is observed:
                    setattr(base, name, original)
            self.closed = True
            self.queue.put(None)
            self.thread.join()
        # On poison/unknown state keep hooks/writer alive for a possible late native return.
        return {'file': self.path.name, 'writer_closed': self.closed,
                'native_instrumented': bool(self.hooks), 'errors': list(self.errors),
                'late_return_possible': not native_finished}


class CallOwner:
    """One serial call owner. Main-thread Stop never interrupts a native worker's join."""

    def __init__(self, name):
        self.name = name
        self.jobs = queue.SimpleQueue()
        self.finished = threading.Event()
        self.cleanup = None  # Atomic Python reference; setting it needs no lock or queue operation.
        self.current = None
        self.last_error = None
        self.thread = threading.Thread(target=self._work, name=f"heater-characterization-{name}", daemon=True)
        self.thread.start()

    def _work(self):
        try:
            while self.cleanup is None:
                try:
                    label, function, result, done = self.jobs.get(timeout=.05)
                except queue.Empty:
                    continue
                self.current = label
                try:
                    if self.cleanup is not None:
                        raise KeyboardInterrupt("Stop requested before queued instrument call")
                    result["value"] = function()
                except BaseException as error:
                    result["error"] = error
                    self.last_error = f"{type(error).__name__}: {error}"
                finally:
                    self.current = None
                    done.set()  # The Python call returned; poison still guards any native timeout.
            self.current = "shutdown"
            self.cleanup()  # Same owner, strictly after the active call, never a second OFF worker.
        except BaseException as error:
            self.last_error = f"{type(error).__name__}: {error}"
        finally:
            self.current = None
            self.finished.set()  # This owner issues no new call; a poisoned native call may still be active.

    def call(self, label, function):
        if self.cleanup is not None:
            raise KeyboardInterrupt("Instrument owner is stopping")
        result, done = {}, threading.Event()
        self.jobs.put((label, function, result, done))
        warned = time.monotonic()
        while not done.wait(.05):
            if self.cleanup is not None:
                raise KeyboardInterrupt("Stop requested; pending I/O remains owned")
            if time.monotonic() - warned >= 5:
                print(f"{self.name}: waiting for {label}; Stop requests safe shutdown, not concurrent Close.")
                warned = time.monotonic()
        if "error" in result:
            raise result["error"]
        return result.get("value")

    def begin_close(self, function):
        if self.cleanup is None:
            self.cleanup = function

    def wait_closed(self, timeout=None, on_interrupt=None, on_progress=None):
        started, warned = time.monotonic(), False
        while not self.finished.is_set():
            try:
                self.finished.wait(.05)
                if on_progress is not None:
                    on_progress()
            except KeyboardInterrupt as error:
                if on_interrupt is not None:
                    on_interrupt(error)
                # The queued OFF is already owned. Repeated Stop cannot interrupt it.
            now = time.monotonic()
            if timeout is not None and now - started >= timeout:
                return self.finished.is_set()
            if on_progress is None and not warned and now - started >= 5 and not self.finished.is_set():
                print(f"{self.name} shutdown pending: waiting for the actual call/verified shutdown. Do not rerun or restart to bypass it.")
                warned = True
        return True

    def status(self):
        return {"owner_finished": self.finished.is_set(), "active_call": self.current,
                "last_error": self.last_error, "thread_id": self.thread.ident}


# Protocol constants: experimental criteria, not manufacturer safety ratings.
TARGETS_C = (30., 40., 50., 60., 70., 80., 90., 100.)
FINAL_TARGET_C = TARGETS_C[-1]
ENVELOPE = {'voltage_v': 22., 'current_a': 10., 'power_w': 50.}
LIMIT_KEYS = {'voltage_v': 'voltage_limit_v', 'current_a': 'current_limit_a', 'power_w': 'power_limit_w'}


def finite(value):
    return type(value) in (int, float) and math.isfinite(value)


class ReachedFinalTarget(Exception):
    pass


class NotReached(Exception):
    pass


class LivePlots:
    def __init__(self):
        import matplotlib.pyplot as plt
        from IPython.display import display
        self.plt = plt
        self.figure, self.axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True, layout='constrained')
        plt.close(self.figure)
        self.handle = display(self.figure, display_id=True)

    def update(self, rows):
        rows = sorted(rows, key=lambda r: r['elapsed_s'])
        x = [r['elapsed_s'] / 60 for r in rows]
        for ax in self.axes:
            ax.clear()
            ax.grid(alpha=.2)
        def values(key):
            return [r.get(key, math.nan) for r in rows]
        self.axes[0].plot(x, values('temperature_c'), label='Measured temperature')
        self.axes[0].plot(x, [r['applied_target_c'] if r['heater_requested'] else math.nan for r in rows], '--', label='Applied setpoint')
        self.axes[0].axhline(FINAL_TARGET_C, color='firebrick', ls=':', lw=1, label='Software stop')
        self.axes[0].set_ylabel('Temperature (°C)')
        self.axes[1].plot(x, values('monitor_output_v'), label='VoltOut monitoring')
        self.axes[1].plot(x, values('monitor_heater_v'), label='VoltHeat monitoring')
        self.axes[1].set_ylabel('Voltage (V)')
        self.axes[2].plot(x, values('monitor_current_a'), label='CurrOut monitoring')
        self.axes[2].set_ylabel('Current (A)')
        self.axes[3].plot(x, values('power_target_w'), label='Power target — NOT measured power')
        self.axes[3].set_ylabel('Target (W)')
        self.axes[3].set_xlabel('Elapsed time (min)')
        for ax in self.axes:
            ax.legend(loc='best', fontsize=8)
        last = rows[-1] if rows else {}
        temperature = last.get('temperature_c')
        shown = f'{temperature:.2f} °C' if finite(temperature) else 'no valid temperature'
        self.figure.suptitle(f"ESI heater — {last.get('phase', 'waiting').replace('_', ' ')} — {shown}")
        self.handle.update(self.figure)

    def save(self, directory, rows):
        self.update(rows)
        self.figure.savefig(directory / 'heater.png', dpi=160)


def load_pure_helper(root, filename):
    """Load only a known pure helper; never execute another notebook."""
    if filename not in ('_experiment_guard.py', '_heater_limits.py'):
        raise ValueError('Unknown pure helper')
    path = root / filename
    raw = path.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    name = '_esibd' + path.stem + '_' + hashlib.sha256(os.path.normcase(str(path.resolve())).encode()).hexdigest()[:16]
    if name in sys.modules:
        module = sys.modules[name]
        if getattr(module, '_source_sha256', None) != digest:
            raise RuntimeError(f'{filename} changed after it was loaded in this kernel. Use a fresh kernel.')
        return module
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        exec(compile(raw, str(path), 'exec'), module.__dict__)
        module._source_sha256 = digest
    except BaseException:
        sys.modules.pop(name, None)
        raise
    return module


def load_experiment_guard(root):
    return load_pure_helper(root, '_experiment_guard.py')

class Experiment:
    def __init__(self, output_dir, factory, *, baseline_s=60., observation_s=60.,
                 stage_s=300., cycle_s=60., cooling_s=180., sample_s=1.,
                 clock=time.monotonic, sleep=time.sleep, plots_factory=LivePlots, native_base=None,
                 run_guard=None, operator_restarts=()):
        self.run_guard = run_guard
        self.operator_restarts = list(operator_restarts)
        self.constructor_attempted = False
        self.native_base, self.trace = native_base, None
        self.output_dir, self.factory = Path(output_dir), factory
        self.baseline_s, self.observation_s, self.stage_s = baseline_s, observation_s, stage_s
        self.cycle_s, self.cooling_s, self.sample_s = cycle_s, cooling_s, sample_s
        self.clock, self.sleep, self.plots_factory = clock, sleep, plots_factory
        self.esi = self.owner = self.plots = self.stream = self.directory = None
        self.connected = self.shutdown_confirmed = self.heat_requested = False
        self.phase, self.requested, self.applied_target = 'preflight', 0., 0.
        self.stop_requests = 0
        self.heat_cancel = threading.Event()
        self.stopping = False
        self.rows, self.errors = [], []
        self.write_lock = threading.Lock()
        self.metadata = {'started_utc': utc(), 'outcome': 'running', 'errors': self.errors,
                         'envelope': dict(ENVELOPE), 'targets_c': TARGETS_C,
                         'baseline_s': baseline_s, 'observation_s': observation_s,
                         'approach_timeout_s': stage_s, 'stage_total_timeout_s': stage_s + observation_s,
                         'final_stage_timeout_s': stage_s, 'cycle_off_s': cycle_s,
                         'cooling_observation_s': cooling_s, 'sample_requested_s': sample_s,
                         'esi_com': ESI_COM, 'esi_baud': ESI_BAUD,
                         'limitations': [
                             'Software temperature stop is sampled: overshoot/inertia remain possible.',
                             'No ADC freshness or simultaneous U/I acquisition is established.',
                             'VoltOut and VoltHeat are distinct monitoring channels; no undocumented comparison to a voltage limit is used.',
                             'Electrical targets and calculated U*I are NOT measured dissipated power.',
                             'Limits remain configured after OFF; no Save/NVM command, power-cycle persistence unknown.',
                             'This test does NOT qualify 175 C, electrical ratings or a cold/safe-to-touch heater.']}

    def validate(self):
        for name in ('baseline_s', 'observation_s', 'stage_s', 'cycle_s', 'cooling_s', 'sample_s'):
            if not finite(getattr(self, name)) or getattr(self, name) <= 0:
                raise ValueError(f'{name} must be finite and positive')

    def claim(self):
        if _HC_GUARD:
            raise RuntimeError('Previous run still owns an uncertain transport; no rerun.')
        if self.run_guard is None:
            raise RuntimeError('Shared ESI experiment guard required before construction')
        self.run_guard.check()
        self.guard_path = self.run_guard.heater_dir / '.heater_characterization.running.json'
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.metadata.update(shutdown_confirmed=False, operator_restarts=self.operator_restarts)
        self.directory = self.output_dir / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + uuid4().hex[:8])
        _HC_GUARD['run'] = self  # Keep partial transitions/locks alive even if claim raises.
        try:
            self.directory.mkdir()
            self.stream = (self.directory / 'samples.csv').open('x', newline='', encoding='utf-8')
            self.writer = csv.DictWriter(self.stream, fieldnames=list(self.empty_row()))
            self.writer.writeheader()
            self.stream.flush()
            os.fsync(self.stream.fileno())
            report_path = self.directory / 'report.json'
            save_json(report_path, self.metadata)
            self.metadata['experiment_claim'] = self.run_guard.claim(self.directory, report_path)
            save_json(report_path, self.metadata)
        except BaseException:
            if self.stream is not None:
                self.stream.close()
            if self.run_guard.abandon_before_hardware():
                _HC_GUARD.clear()
            raise

    def poison_state(self):
        try:
            state = self.esi._transport_poisoned
            return state if type(state) is bool else None
        except BaseException:
            return None

    def safe_transport(self):
        if self.poison_state() is not False:
            raise RuntimeError('Transport poisoned or unknown: no further ESI call, OFF or Close; use physical safety controls.')

    def call(self, method, *args, cleanup=False, **kwargs):
        if not cleanup and method in ('collect_diagnostics', 'set_heater_temperature', 'set_output_active', 'configure_heat_limits'):
            kwargs['cancel_event'] = self.heat_cancel
        def invoke():
            self.safe_transport()
            if not cleanup and (self.stopping or self.stop_requests):
                raise KeyboardInterrupt('Stop requested before instrument call')
            self.trace.record('operation_start', method=method, phase=self.phase, requested_target_c=self.requested)
            try:
                result = getattr(self.esi, method)(*args, **kwargs)
            except BaseException as error:
                self.trace.record('operation_exception', method=method, exception_type=type(error).__name__, message=str(error))
                if not cleanup and isinstance(error, InterruptedError) and self.heat_cancel.is_set():
                    raise KeyboardInterrupt('Stop cancelled the pending heater operation') from error
                raise
            self.trace.record('operation_return', method=method)
            if method == 'connect' and result is True:
                self.connected = True  # Preserve a late connection success on Stop.
            self.safe_transport()
            return result
        if self.owner is not None and threading.get_ident() == self.owner.thread.ident:
            return invoke()
        return self.owner.call(method, invoke)

    def request_stop(self):
        self.heat_cancel.set()
        self.stopping = True
        if self.owner is not None:
            self.owner.begin_close(self.cleanup)

    def interrupt(self, error):
        self.errors.append(f'{type(error).__name__}: {error}')
        self.metadata['outcome'] = 'interrupted'

    def check_config(self, config):
        for short, key in LIMIT_KEYS.items():
            value, maximum = config[key], config['hardware_limits']['max_' + short]
            if not finite(value) or not finite(maximum) or not 0 < value <= min(ENVELOPE[short], maximum):
                raise RuntimeError(f'Applied {short} outside the verified envelope')
            expected = self.metadata.get('applied_configuration', {}).get(key)
            if expected is not None and value != expected:
                raise RuntimeError(f'Electrical limit changed: {key}')
        maximum = config['hardware_limits']['max_temperature_c']
        if not finite(maximum) or maximum < FINAL_TARGET_C:
            raise RuntimeError('Hardware temperature limit does not cover this program')

    def check_snapshot(self, diagnostic):
        heat = diagnostic['heat']
        if diagnostic['main_state']['name'] != 'STATE_ON' or int(diagnostic['device_state']['hex'], 16) != 0:
            raise RuntimeError('Controller fault/not ready; no interlock bypass')
        for address in (1, 2):
            hv = diagnostic['modules'][address]
            if (hv['module_active'] is not False or hv['module_gate_active'] is not False
                    or not finite(hv['target_v']) or hv['target_v'] != 0):
                raise RuntimeError(f'HV{address} is not confirmed OFF with zero target')
        for key in ('monitor_temperature_c', 'monitor_output_v', 'monitor_voltage_v', 'monitor_current_a',
                    'output_voltage_v', 'heater_power_w'):
            if not finite(heat[key]):
                raise RuntimeError(f'Invalid diagnostic field: {key}')
        maximum = heat['hardware_limits']['max_temperature_c']
        if heat['valid'] is not True or not finite(maximum) or not 0 <= heat['monitor_temperature_c'] <= maximum:
            raise RuntimeError('Invalid temperature sensor or hardware temperature range')
        if (heat['module_active'] is not self.heat_requested
                or heat['module_gate_active'] is not self.heat_requested
                or (self.heat_requested and heat['active'] is not True)):
            raise RuntimeError('Heater activation/regulation not confirmed')
        if not finite(heat['target_temperature_c']) or heat['target_temperature_c'] != self.applied_target:
            raise RuntimeError('Heater target changed or unverified')
        if 'applied_configuration' in self.metadata:
            self.check_config(heat)

    def empty_row(self):
        return {'timestamp_utc': utc(), 'elapsed_s': self.clock() - self.started,
                'call_started_s': math.nan, 'call_finished_s': math.nan,
                'phase': self.phase, 'heater_requested': self.heat_requested,
                'requested_target_c': self.requested, 'applied_target_c': self.applied_target,
                'temperature_c': math.nan, 'monitor_output_v': math.nan,
                'monitor_heater_v': math.nan, 'monitor_current_a': math.nan,
                'voltage_target_v': math.nan, 'power_target_w': math.nan,
                'u_heat_times_i_calculated_w': math.nan,
                'voltage_limit_v': math.nan, 'current_limit_a': math.nan, 'power_limit_w': math.nan,
                'diagnostic_json': '', 'error': ''}

    def append(self, row):
        with self.write_lock:
            self.writer.writerow(row)
            self.stream.flush()
            self.rows.append(row)

    def sample(self, cleanup=False):
        row = self.empty_row()
        problem = None
        try:
            row['call_started_s'] = self.clock() - self.started
            diagnostic = self.call('collect_diagnostics', cleanup=cleanup)
            row['call_finished_s'] = self.clock() - self.started
            row['timestamp_utc'], row['elapsed_s'] = utc(), row['call_finished_s']
            row['diagnostic_json'] = json.dumps(json_safe(diagnostic), allow_nan=False)
            heat = diagnostic['heat']
            self.check_snapshot(diagnostic)
            row.update(temperature_c=heat['monitor_temperature_c'], monitor_output_v=heat['monitor_output_v'],
                       monitor_heater_v=heat['monitor_voltage_v'], monitor_current_a=heat['monitor_current_a'],
                       voltage_target_v=heat['output_voltage_v'], power_target_w=heat['heater_power_w'],
                       u_heat_times_i_calculated_w=heat['monitor_voltage_v'] * heat['monitor_current_a'])
            for key in LIMIT_KEYS.values():
                row[key] = heat[key]
            if not cleanup and heat['monitor_temperature_c'] >= FINAL_TARGET_C:
                # Cleanup changes phase/targets/time: classify this observation BEFORE queuing it.
                final_stage_observed = (row['phase'] == 'heating' and row['requested_target_c'] == FINAL_TARGET_C
                                        and row['call_finished_s'] <= self.stage_deadline - self.started)
                self.request_stop()  # OFF queued BEFORE storage/plotting; no further heating command.
                if final_stage_observed:
                    raise ReachedFinalTarget(f'{FINAL_TARGET_C:g} C observed: heater OFF requested, no final hold')
                raise RuntimeError(f'{FINAL_TARGET_C:g} C software threshold reached outside the final-stage time window')
        except BaseException as error:
            problem = error
            if not cleanup:
                self.request_stop()
            if not isinstance(error, ReachedFinalTarget):
                row['error'] = f'{type(error).__name__}: {error}'
        self.append(row)
        if problem is not None:
            raise problem
        return row

    def pause(self, duration, cleanup=False):
        deadline = self.clock() + duration
        while self.clock() < deadline:
            if cleanup and self.stop_requests > 1:
                self.metadata['cooling_observation_shortened_by_stop'] = True
                break  # A repeated Stop shortens observation, never interrupts verified disconnect.
            if not cleanup and (self.stopping or self.stop_requests):
                raise KeyboardInterrupt('Stop requested')
            polled_at = self.clock()
            self.sample(cleanup=cleanup)
            if not cleanup and self.phase == 'baseline' and self.rows[-1]['temperature_c'] >= 30:
                raise RuntimeError('Start below 30 C; the initial 30 C restart test must not be skipped')
            if not cleanup:
                self.refresh_plot()
            self.sleep(max(0., min(self.sample_s - (self.clock() - polled_at), deadline - self.clock())))

    def refresh_plot(self):
        if self.clock() - getattr(self, 'last_plot', -math.inf) >= 5:
            self.last_plot = self.clock()
            self.plots.update(list(self.rows))

    def cleanup_progress(self):
        # Display-only: no instrument calls and no change to the owner's cleanup.
        phase = self.phase if self.phase in ('cooling', 'shutdown') else 'pending'
        if (not self.owner.finished.is_set()
                and phase != getattr(self, 'cleanup_display_phase', None)
                and not getattr(self, 'cleanup_display_failed', False)):
            self.cleanup_display_phase = phase
            messages = {
                'pending': 'Waiting for the owned operation and heater OFF confirmation; do not rerun.',
                'cooling': f'Cooling observation (up to {self.cooling_s:g} s); not a cold/safe-to-touch confirmation.',
                'shutdown': 'Verifying HV discharge and port closure; shutdown is not yet confirmed.',
            }
            try:
                print(messages[phase])
            except Exception as error:
                self.cleanup_display_failed = True
                self.errors.append(f'Cleanup display: {type(error).__name__}: {error}')
        # Main-thread rendering only; a plotting error cannot abort the owner's OFF/disconnect.
        if not getattr(self, 'cleanup_plot_failed', False):
            try:
                self.refresh_plot()
            except Exception as error:
                self.cleanup_plot_failed = True
                self.errors.append(f'Cleanup plot: {type(error).__name__}: {error}')

    def preflight(self):
        if self.call('connect') is not True:
            raise RuntimeError('Connection unconfirmed')
        diagnostic = self.call('collect_diagnostics')
        self.metadata['initial_diagnostics'] = diagnostic
        self.check_snapshot(diagnostic)
        if diagnostic['heat']['monitor_temperature_c'] >= 30:
            raise RuntimeError('Start cold below 30 C; no initial stage is skipped')
        self.metadata['initial_configuration'] = self.call('get_heat_configuration')
        self.phase = 'baseline'
        self.pause(self.baseline_s)
        config = self.call('get_heat_configuration')  # Fresh bounds immediately before configuration.
        self.metadata['configuration_before_limits'] = config
        maximum = config['hardware_limits']['max_temperature_c']
        if not finite(maximum) or maximum < FINAL_TARGET_C:
            raise RuntimeError('Hardware temperature limit below the program')
        wanted = {}
        for short in LIMIT_KEYS:
            maximum = config['hardware_limits']['max_' + short]
            if not finite(maximum) or maximum <= 0:
                raise RuntimeError('Invalid hardware electrical limit')
            wanted[short] = bounded_limit(min(ENVELOPE[short], maximum), short == 'power_w')
        self.metadata['requested_limits'] = wanted
        echoes = self.call('configure_heat_limits', **wanted)
        self.metadata['applied_limit_echoes'] = echoes
        if not isinstance(echoes, dict) or set(echoes) != set(LIMIT_KEYS):
            raise RuntimeError('Missing applied limit echoes')
        readback = self.call('get_heat_configuration')
        self.check_config(readback)
        for short, key in LIMIT_KEYS.items():
            if not finite(echoes[short]) or echoes[short] != wanted[short] or readback[key] != echoes[short]:
                raise RuntimeError('Limit echo/readback/native-code mismatch; no activation')
        self.metadata['applied_configuration'] = readback
        save_json(self.directory / 'report.json', self.metadata)
        print('Applied limits:', wanted, '— unchanged throughout the experiment.')

    def heat_stage(self, target, repeat=False):
        self.sample()  # Current HV gates, sensor, target, limits and activation before a new demand.
        self.phase = 'restart_30' if repeat else 'heating'
        approach_deadline = self.clock() + self.stage_s
        self.stage_deadline = approach_deadline
        self.requested = target
        self.applied_target = self.call('set_heater_temperature', target)
        if not finite(self.applied_target) or not 0 < self.applied_target <= FINAL_TARGET_C or abs(self.applied_target - target) > 1:
            raise RuntimeError('Applied temperature outside the approved approach window/program')
        if not self.heat_requested:
            self.heat_requested = True
            if self.call('set_output_active', 0, True) is not True:
                raise RuntimeError('Heater ON unconfirmed')
        print(f'{self.phase}: {target:g} °C, approach within {self.stage_s:g} s; fixed electrical envelope.')
        first_in_window = None
        approached = False
        while self.clock() < self.stage_deadline:
            polled_at = self.clock()
            row = self.sample()
            if self.clock() >= self.stage_deadline:
                break
            if target < FINAL_TARGET_C and abs(row['temperature_c'] - target) <= 1:
                if not approached:
                    approached = True
                    # One fixed extension: re-entry never moves the absolute deadline.
                    self.stage_deadline = approach_deadline + self.observation_s
                    print(f'{target:g} °C zone reached: require {self.observation_s:g} s continuously within ±1 °C; total stage limit {self.stage_s + self.observation_s:g} s.')
                # Qualify returned samples, not time spent subsequently storing/plotting them.
                observed_at = row['call_finished_s']
                if first_in_window is None:
                    first_in_window = observed_at
                if observed_at - first_in_window >= self.observation_s:
                    self.metadata.setdefault('observed_stages', []).append({'target_c': target, 'repeat': repeat, 'elapsed_s': row['elapsed_s']})
                    return
            else:
                first_in_window = None
            self.refresh_plot()
            self.sleep(max(0., min(self.sample_s - (self.clock() - polled_at), self.stage_deadline - self.clock())))
        self.request_stop()
        criterion = (f'{self.observation_s:g} s continuous observation within the {self.stage_s + self.observation_s:g} s total stage limit'
                     if approached else f'approach within {self.stage_s:g} s')
        raise NotReached(f'{target:g} C: {criterion} not completed under the fixed envelope; no power increase')

    def heater_off(self, cleanup=False):
        if self.call('set_output_active', 0, False, cleanup=cleanup) is not False:
            raise RuntimeError('Heater OFF unconfirmed')
        self.heat_requested, self.requested, self.applied_target = False, 0., 0.
        diagnostic = self.call('collect_diagnostics', cleanup=cleanup)
        self.check_snapshot(diagnostic)
        return diagnostic

    def restart_test(self):
        before = self.call('get_heat_configuration')
        self.check_config(before)
        self.metadata['cycle_before'] = before
        self.heater_off()
        self.phase = 'cycle_off'
        self.pause(self.cycle_s)
        after = self.call('get_heat_configuration')
        self.metadata['cycle_after'] = after
        self.check_config(after)  # No rewrite may mask a reset to zero.
        self.heat_stage(30., repeat=True)
        self.metadata['restart_30_confirmed'] = True

    def discharge_observation(self, state):
        self.metadata['last_discharge'] = state
        # This callback only copies the provided observation: no getter or other instrument call.
        self.trace.record('discharge_observation', state=state)

    def cleanup(self):
        # ONLY the instrument owner executes this, after the current call returns.
        if self.esi is None:
            self.shutdown_confirmed = not self.constructor_attempted
            return
        try:
            self.safe_transport()
            if self.connected:
                self.metadata['off_diagnostics'] = self.heater_off(cleanup=True)
                self.phase = 'cooling'
                cooling_started = self.clock()
                try:
                    self.pause(self.cooling_s, cleanup=True)
                finally:
                    self.metadata['cooling_observed_s'] = self.clock() - cooling_started
                self.metadata['final_configuration'] = self.call('get_heat_configuration', cleanup=True)
                self.check_config(self.metadata['final_configuration'])
        except BaseException as error:
            self.errors.append(f'{type(error).__name__}: {error}')
        finally:
            try:
                self.safe_transport()
                self.phase = 'shutdown'
                closed = self.call('disconnect', cleanup=True,
                                   on_discharge=self.discharge_observation)
                self.shutdown_confirmed = closed is True and self.connected
                if not self.shutdown_confirmed:
                    raise RuntimeError('OFF/discharge/port closure unconfirmed for an established connection')
            except BaseException as error:
                self.errors.append(f'{type(error).__name__}: {error}')

    def run(self):
        previous = None
        if threading.current_thread() is threading.main_thread():
            previous = signal.getsignal(signal.SIGINT)
            def stop_signal(signum, frame):
                self.stop_requests += 1
                self.request_stop()
                raise KeyboardInterrupt('Stop requested; no concurrent OFF/Close')
            signal.signal(signal.SIGINT, stop_signal)
        try:
            return self.sequence()
        finally:
            if previous is not None:
                signal.signal(signal.SIGINT, previous)
            if self.run_guard is not None and not self.constructor_attempted and _HC_GUARD:
                if self.run_guard.abandon_before_hardware():
                    _HC_GUARD.clear()

    def sequence(self):
        self.validate()
        self.started = self.clock()
        self.claim()
        try:
            self.trace = NativeTrace(self.directory)
            self.plots = self.plots_factory()
            self.owner = CallOwner('ESI')
            def construct():
                if self.stopping or self.stop_requests:
                    raise KeyboardInterrupt('Stopped before construction')
                self.constructor_attempted = True
                self.run_guard.mark_hardware_started()
                self.esi = self.factory(self.directory)
                if self.native_base is not None:
                    self.trace.install(self.native_base, object.__getattribute__(self.esi, '_backend'))
            self.owner.call('construct', construct)
            self.preflight()
            for target in TARGETS_C:
                self.heat_stage(target)
                if target == 30:
                    self.restart_test()
        except ReachedFinalTarget as error:
            self.metadata['outcome'] = 'reached_100_no_hold'
            self.metadata['completion'] = str(error)
        except NotReached as error:
            self.metadata['outcome'] = 'not_reached'
            self.metadata['completion'] = str(error)
        except BaseException as error:
            self.request_stop()
            self.errors.append(f'{type(error).__name__}: {error}')
            self.metadata['outcome'] = 'interrupted' if isinstance(error, KeyboardInterrupt) else 'error'
        finally:
            self.request_stop()
            if self.owner is not None:
                self.owner.wait_closed(on_interrupt=self.interrupt, on_progress=self.cleanup_progress)
            else:
                self.shutdown_confirmed = self.esi is None
            if self.trace is not None:
                self.metadata['native_trace'] = self.trace.finish(self.esi is None or self.poison_state() is False)
            valid_rows = sorted((r for r in self.rows if finite(r['temperature_c'])), key=lambda r: r['elapsed_s'])
            intervals = [b['elapsed_s'] - a['elapsed_s'] for a, b in zip(valid_rows, valid_rows[1:])]
            self.metadata['observations'] = {
                'maximum_temperature_c': max((r['temperature_c'] for r in valid_rows), default=None),
                'last_temperature_c': valid_rows[-1]['temperature_c'] if valid_rows else None,
                'maximum_sample_interval_s': max(intervals, default=None),
                'maximum_monitor_output_v': max((r['monitor_output_v'] for r in valid_rows), default=None),
                'maximum_monitor_heater_v': max((r['monitor_heater_v'] for r in valid_rows), default=None),
                'maximum_monitor_current_a': max((r['monitor_current_a'] for r in valid_rows), default=None),
                'maximum_power_target_w_NOT_measured': max((r['power_target_w'] for r in valid_rows), default=None)}
            self.metadata.update(ended_utc=utc(), samples=len(self.rows),
                                 shutdown_confirmed=self.shutdown_confirmed,
                                 transport_poisoned=self.poison_state() if self.esi is not None else None,
                                 owner=self.owner.status() if self.owner is not None else None)
            if not self.shutdown_confirmed:
                self.metadata['outcome'] = 'shutdown_unconfirmed'
                print('SHUTDOWN UNCONFIRMED: heater/HV may remain energized. Use physical safety controls; rerun blocked.')
            elif self.errors and self.metadata['outcome'] == 'reached_100_no_hold':
                self.metadata['outcome'] = 'error'
            if self.stream is not None:
                self.stream.close()
            save_json(self.directory / 'report.json', self.metadata)
            try:
                if self.plots is not None:
                    self.plots.save(self.directory, self.rows)
            except BaseException as error:
                self.errors.append(f'{type(error).__name__}: {error}')
                save_json(self.directory / 'report.json', self.metadata)
            if self.constructor_attempted:
                if self.run_guard.finalize(
                        shutdown_confirmed=self.shutdown_confirmed,
                        owners_idle=self.owner is not None and self.owner.finished.is_set(),
                        auxiliary_closed=True,
                        transport_poisoned=self.poison_state() if self.esi is not None else None):
                    _HC_GUARD.clear()
            print(self.metadata['outcome'], '→', self.directory)
            print('Limits remain configured; no NVM save requested. This does not qualify 175 C or a cold heater.')
        return self


def main():
    global bounded_limit
    guard_path = OUTPUT_DIR / '.heater_characterization.running.json'
    if not ARM_HEATING:
        print('Heating NOT armed — no hardware accessed. Close Explorer/CGC utilities; use a fresh 64-bit Windows kernel.')
        print('Protocol: 30/40/50/60/70/80/90/100 C, fixed <=22 V / 10 A / 50 W; OFF/ON at 30 C; OFF at first observed >=100 C.')
        print('Stop: interrupt the kernel once; wait for verified OFF/discharge. Never rerun to bypass an uncertain shutdown.')
        if _HC_GUARD or guard_path.exists():
            print('Unfinished run remains: ARM=False does not turn hardware OFF.')
        return None
    if _HC_GUARD:
        raise RuntimeError('Unfinished run in this kernel: no operator recovery here; terminate the old process.')
    if sys.platform != 'win32' or sys.maxsize <= 2**32:
        raise RuntimeError('Use 64-bit Windows Python; no hardware accessed.')
    if type(ESI_COM) is not int or not 1 <= ESI_COM <= 255 or ESI_BAUD != 230400:
        raise ValueError('Use the actual COM number (1..255) and 230400 baud; no fallback.')
    root = find_plugin(PLUGIN_DIR)
    output_dir = root / 'logs' / 'esi_heater_characterization'
    if guard_path.exists() and guard_path.resolve() != (output_dir / guard_path.name).resolve():
        raise RuntimeError('Unfinished run at the previous output path: do not bypass it by changing directories.')
    fingerprints = runtime_fingerprints(root)
    helper = load_experiment_guard(root)
    ownership = helper.ExperimentGuard(
        plugin_dir=root, com=ESI_COM, kind='heater', output_dir=output_dir,
        kernel_globals=globals(), legacy_pt_dirs=(NOTEBOOK_DIR / 'pressure_temperature_runs',))
    run = None
    try:
        records = ownership.authorize_restart(input_fn=input)
        bounded_limit = load_pure_helper(root, '_heater_limits.py').bounded_limit
        runtime = load_private(root / 'vendor/runtime/__init__.py', package=True, fresh=True)
        require_safe_esi(runtime)
        run = Experiment(output_dir, lambda directory: runtime.ESI(
            'heater-characterization', ESI_COM, baudrate=ESI_BAUD, log_dir=directory, process_backend=False),
            native_base=sys.modules[runtime.__name__ + '.esi.esi_base'].ESIBase,
            run_guard=ownership, operator_restarts=records)
        run.metadata['runtime_sources'] = fingerprints
        return run.run()
    finally:
        if run is None:
            ownership.abandon_before_hardware()  # No constructor attempted; historical evidence stays intact.


main()
